# Netflix Data Cleaning & Preprocessing
**Role**: Senior Data Engineer / Senior Data Analyst  
**Project**: End-to-End Netflix Data Analysis  

This notebook performs professional-grade data cleaning on the raw Netflix dataset (`netflix_titles.csv`). 
We will handle missing values, duplicates, standardise text, parse dates, convert durations into numeric features, and prepare the dataset for SQL database ingestion and Power BI reporting.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries successfully imported!")

## 1. Load the Raw Data

In [ ]:
raw_data_path = '../data/netflix_titles.csv'
df = pd.read_csv(raw_data_path)

# Display first 5 rows
df.head()

In [ ]:
# Dataset Info and Null summary
print(df.info())
print("\nMissing values count per column:")
print(df.isnull().sum())

## 2. Data Cleaning Tasks
We will address the following:
- Trim whitespaces from string columns.
- Clean missing values in `director`, `cast`, `country`, `rating`, and `date_added`.
- Check for and drop duplicate rows.
- Standardize categorical fields.
- Parse `date_added` and extract `year_added`, `month_added`, and `day_added`.
- Parse `duration` into numeric format (minutes for Movies, seasons for TV Shows).

### 2.1 Trim Whitespace & Handle Missing Values

In [ ]:
# Trim whitespaces from string columns
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

# Handle text missing values
df['director'] = df['director'].replace({'nan': 'Unknown', '': 'Unknown'}).fillna('Unknown')
df['cast'] = df['cast'].replace({'nan': 'Unknown', '': 'Unknown'}).fillna('Unknown')
df['country'] = df['country'].replace({'nan': 'Unknown', '': 'Unknown'}).fillna('Unknown')
df['rating'] = df['rating'].replace({'nan': 'Unknown', '': 'Unknown'}).fillna('Unknown')

# Drop rows with missing date_added since they are very few (< 10 rows)
df = df[df['date_added'] != 'nan']
df = df.dropna(subset=['date_added'])

print("Remaining missing values:")
print(df.isnull().sum())

### 2.2 Standardise Type Column

In [ ]:
# Standardize categories to 'Movie' and 'TV Show'
type_lower = df['type'].str.lower().str.strip()
df['type'] = type_lower.map({'movie': 'Movie', 'tv show': 'TV Show', 'tv-show': 'TV Show'}).fillna('Unknown')
print("Content types after standardization:", df['type'].unique())

### 2.3 Date Parsing and Feature Engineering

In [ ]:
# Convert date_added to datetime format
df['date_added_dt'] = pd.to_datetime(df['date_added'], errors='coerce', format='%B %d, %Y')

# For dates that did not match format, use fallback parser
if df['date_added_dt'].isnull().sum() > 0:
    df.loc[df['date_added_dt'].isnull(), 'date_added_dt'] = pd.to_datetime(df['date_added'], errors='coerce')

# Drop any remaining unparseable dates
df = df.dropna(subset=['date_added_dt'])

# Extract new date-based features
df['year_added'] = df['date_added_dt'].dt.year.astype(int)
df['month_added'] = df['date_added_dt'].dt.strftime('%B')
df['day_added'] = df['date_added_dt'].dt.day.astype(int)

# Convert date_added to MySQL DATE format: YYYY-MM-DD
df['date_added'] = df['date_added_dt'].dt.strftime('%Y-%m-%d')

# Remove temporary column
df = df.drop(columns=['date_added_dt'])
print("Date added columns engineered:")
df[['date_added', 'year_added', 'month_added', 'day_added']].head()

### 2.4 Parse Duration Column
Movies have durations in 'min' (minutes) and TV Shows in 'Season(s)'. We will parse these and split them into distinct numeric fields: `duration_minutes` and `duration_seasons`.

In [ ]:
df['duration_minutes'] = np.nan
df['duration_seasons'] = np.nan

for idx, row in df.iterrows():
    duration_val = str(row['duration']).lower()
    if 'min' in duration_val:
        minutes = int(''.join(filter(str.isdigit, duration_val)))
        df.at[idx, 'duration_minutes'] = minutes
    elif 'season' in duration_val:
        seasons = int(''.join(filter(str.isdigit, duration_val)))
        df.at[idx, 'duration_seasons'] = seasons

# Cast numeric columns to Pandas nullable Int64
df['duration_minutes'] = df['duration_minutes'].astype('Int64')
df['duration_seasons'] = df['duration_seasons'].astype('Int64')

print("Duration columns after parsing:")
df[['type', 'duration', 'duration_minutes', 'duration_seasons']].sample(5)

### 2.5 Remove Duplicates & Verify Final Cleaned Data

In [ ]:
# Identify and drop duplicate records
initial_rows = df.shape[0]
df = df.drop_duplicates()
print(f"Dropped {initial_rows - df.shape[0]} duplicates.")

# Clean up order by sorting show_id numerically
df['id_num'] = df['show_id'].str.extract(r'(\d+)').astype(int)
df = df.sort_values(by='id_num').drop(columns=['id_num']).reset_index(drop=True)

print(f"Final clean data shape: {df.shape}")
df.head()

## 3. Export Cleaned Dataset

In [ ]:
output_path = '../data/cleaned_netflix.csv'
df.to_csv(output_path, index=False)
print(f"Successfully saved clean dataset to: {output_path}")

# Save normalized tables for relational SQL database ingestion
output_dir = '../data'
titles_cols = [
    'show_id', 'type', 'title', 'date_added', 'release_year', 'rating',
    'duration', 'duration_minutes', 'duration_seasons', 'description',
    'year_added', 'month_added', 'day_added'
]
titles_df = df.drop(columns=['director', 'cast', 'country', 'listed_in'])[titles_cols]
titles_df.to_csv(os.path.join(output_dir, 'normalized_titles.csv'), index=False)

def split_and_explode(dataframe, column_name, target_col_name):
    split_df = dataframe[['show_id', column_name]].copy()
    split_df[column_name] = split_df[column_name].astype(str).str.split(',')
    exploded = split_df.explode(column_name)
    exploded[column_name] = exploded[column_name].str.replace('"', '', regex=False)
    exploded = exploded[exploded[column_name].notna() & (exploded[column_name] != '') & (exploded[column_name] != 'Unknown')]
    return exploded.rename(columns={column_name: target_col_name}).drop_duplicates()

split_and_explode(df, 'director', 'director').to_csv(os.path.join(output_dir, 'normalized_directors.csv'), index=False)
split_and_explode(df, 'cast', 'actor').to_csv(os.path.join(output_dir, 'normalized_cast.csv'), index=False)
split_and_explode(df, 'country', 'country').to_csv(os.path.join(output_dir, 'normalized_countries.csv'), index=False)
split_and_explode(df, 'listed_in', 'genre').to_csv(os.path.join(output_dir, 'normalized_genres.csv'), index=False)
print("Normalized dimension tables successfully exported for SQL!")